# Bank Marketing Campaign & Customer Segmentation Analysis

## 1. Project objective and questions

**Business problem:** a (Portuguese) bank runs telephone marketing campaigns to sell a **term deposit**. Calls are expensive,
so the bank wants to know *who* is more likely to subscribe and *how* campaigns should be run.

**Dataset:** UCI *Bank Marketing* (`bank-full.csv`): 45,211 contacts, 16 input columns plus the target `y`
(did the client subscribe to a term deposit: yes/no).

**Questions this notebook answers**

1. What percentage of customers subscribed?
2. Which age groups are more likely to subscribe?
3. Does job type influence subscription?
4. Does education level influence the response?
5. Does marital status relate to subscription?
6. Does account balance differ between subscribers and non-subscribers?
7. How does the number of campaign contacts affect the outcome?
8. Which communication method appears most effective?
9. Does previous campaign history relate to the current outcome?
10. Which customer segments show interesting patterns?

**Method:** data understanding -> cleaning -> univariate, bivariate and multivariate analysis -> segmentation -> recommendations.
Run the cells top to bottom (*Kernel -> Restart & Run All*). Charts, the EDA report (PDF) and the presentation are saved into `reports/`.

> **How to read the numbers.** "Subscription rate" = share of contacted customers with `y = yes`. We compare each group's rate with the
> overall rate. Differences are **associations, not proof of cause**. Error bars on bar charts are approximate 95% intervals.

## 2. Importing libraries and loading the data

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

cwd = Path.cwd()
PROJECT_DIR = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = PROJECT_DIR / "data"
REPORTS_DIR = PROJECT_DIR / "reports"
REPORTS_DIR.mkdir(exist_ok=True); DATA_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 150, "axes.titleweight": "bold", "axes.titlesize": 13,
                     "axes.spines.top": False, "axes.spines.right": False, "grid.color": "#e6e6e6"})
BLUE, GREY, ORANGE = "#2a6fb0", "#9aa5b1", "#d9822b"
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 40

def save_fig(fig, name):
    path = REPORTS_DIR / name
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print("Saved chart ->", path.relative_to(PROJECT_DIR))

results = {}   # key numbers are collected here and reused by the report / slides

def _fetch_zip(url, dest_dir):
    """Download a zip from UCI and extract it (also unzips zips inside the zip). Returns True on success."""
    import urllib.request, zipfile, io
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        data = urllib.request.urlopen(req, timeout=120).read()
        zf = zipfile.ZipFile(io.BytesIO(data)); zf.extractall(dest_dir)
        for inner in list(Path(dest_dir).glob("*.zip")):
            zipfile.ZipFile(inner).extractall(dest_dir)
        return True
    except Exception as e:
        print("Automatic download failed:", e)
        return False

DATA_PATH = DATA_DIR / "bank-full.csv"
if not DATA_PATH.exists():
    print("bank-full.csv not found - trying to download it from UCI ...")
    _fetch_zip("https://archive.ics.uci.edu/static/public/222/bank+marketing.zip", DATA_DIR)
if not DATA_PATH.exists():
    raise FileNotFoundError("Please download the dataset from https://archive.ics.uci.edu/dataset/222/bank+marketing, "
                            "unzip it (and the inner bank.zip) and put 'bank-full.csv' in the data folder.")

df_raw = pd.read_csv(DATA_PATH, sep=";")      # this file uses semicolons
print("Loaded:", DATA_PATH.name)
df_raw.head()

## 3. Data understanding

`df_raw` is the original data and is **never modified**; all later steps use a copy.

### Data dictionary

| Column | Type | Meaning |
|---|---|---|
| `age` | number | Age of the client |
| `job` | category | Type of job (e.g. admin., blue-collar, student, retired) |
| `marital` | category | Marital status (married / single / divorced) |
| `education` | category | primary / secondary / tertiary / unknown |
| `default` | yes/no | Has credit in default? |
| `balance` | number | Average yearly balance (euros); can be negative |
| `housing` | yes/no | Has a housing loan? |
| `loan` | yes/no | Has a personal loan? |
| `contact` | category | Contact channel: cellular / telephone / unknown |
| `day`, `month` | number / text | Day and month of the **last contact** of the current campaign |
| `duration` | number | Duration of the last call in seconds - **only known after the call** |
| `campaign` | number | Number of contacts in *this* campaign for this client (including the last one) |
| `pdays` | number | Days since the client was last contacted in a *previous* campaign (-1 = never contacted) |
| `previous` | number | Number of contacts before this campaign |
| `poutcome` | category | Outcome of the previous campaign (success / failure / other / unknown) |
| `y` | yes/no | **Target:** did the client subscribe to a term deposit? |

In [ ]:
print(f"Rows: {df_raw.shape[0]:,}   Columns: {df_raw.shape[1]}")
display(df_raw.dtypes.to_frame("dtype").T)
display(df_raw.describe().T)
display(df_raw.describe(include="object").T)

In [ ]:
missing = df_raw.isna().sum()
unknown = (df_raw.select_dtypes("object") == "unknown").sum()
quality = pd.DataFrame({"missing (NaN)": missing, "'unknown' labels": unknown.reindex(missing.index).fillna(0).astype(int)})
display(quality[(quality > 0).any(axis=1)] if (quality > 0).any().any() else quality.head(0))
n_dupes = int(df_raw.duplicated().sum())
print(f"Total NaN cells: {int(missing.sum())}   |   Exact duplicate rows: {n_dupes}")
results.update(raw_rows=int(df_raw.shape[0]), raw_cols=int(df_raw.shape[1]), duplicate_rows=n_dupes, missing_cells=int(missing.sum()),
               unknown_counts={k: int(v) for k, v in unknown.items() if v > 0})

**What to notice:** the dataset has no true missing values, but several columns use the label **`unknown`** as a hidden missing value
(notably `poutcome` and `contact`, and a few in `job` and `education`). These are kept as their own category rather than dropped,
because *not knowing* is itself informative here (e.g. `poutcome = unknown` usually means the client was never contacted before).

## 4. Data cleaning and feature preparation

Cleaning decisions (documented):

* Remove exact duplicate rows (none are expected).
* Keep `unknown` as a category (see above).
* Create `y_bin` (1 = subscribed, 0 = not) for rates and correlations.
* Create `age_group`, `balance_band` and `campaign_group` to make patterns easier to read.
* `pdays = -1` means "never contacted before": we create `previously_contacted` and a cleaned `pdays_clean` (NaN when never contacted) so -1 does not distort statistics.
* Convert text columns to categorical types; order `month` from January to December.
* **Outliers are investigated, not deleted** (see below).

In [ ]:
df = df_raw.drop_duplicates().copy()
df["y_bin"] = (df["y"] == "yes").astype(int)

MONTHS = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
df["month"] = pd.Categorical(df["month"], categories=MONTHS, ordered=True)

df["age_group"] = pd.cut(df["age"], bins=[0, 24, 34, 44, 54, 64, 120],
                         labels=["Under 25", "25–34", "35–44", "45–54", "55–64", "65+"])
df["balance_band"] = pd.cut(df["balance"], bins=[-np.inf, 0, 500, 2000, 5000, np.inf],
                            labels=["≤ €0", "€1–500", "€501–2,000", "€2,001–5,000", "> €5,000"])
df["campaign_group"] = pd.cut(df["campaign"], bins=[0, 1, 2, 3, 5, 10, np.inf], labels=["1", "2", "3", "4–5", "6–10", "11+"])
df["previously_contacted"] = np.where(df["pdays"] == -1, "never contacted before", "contacted before")
df["pdays_clean"] = df["pdays"].replace(-1, np.nan)
for c in ["job", "marital", "education", "default", "housing", "loan", "contact", "poutcome", "y"]:
    df[c] = df[c].astype("category")

print(f"Rows after removing duplicates: {len(df):,}")
display(df[["age", "age_group", "balance", "balance_band", "campaign", "campaign_group", "pdays", "pdays_clean", "y_bin"]].head())
results["clean_rows"] = int(len(df))

### Outlier detection

We use the IQR rule (values above Q3 + 1.5 x IQR, or below Q1 - 1.5 x IQR) only to **flag** unusual values, then look at them.

In [ ]:
num_cols = ["age", "balance", "duration", "campaign", "previous"]
rows = []
for c in num_cols:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = (df[c] < lo) | (df[c] > hi)
    rows.append({"variable": c, "lower fence": lo, "upper fence": hi, "outliers": int(flagged.sum()),
                 "% of rows": flagged.mean() * 100, "min": df[c].min(), "max": df[c].max()})
outlier_tbl = pd.DataFrame(rows).set_index("variable"); display(outlier_tbl)
results["outliers"] = {k: {"n": int(v["outliers"]), "pct": float(v["% of rows"]), "max": float(v["max"])} for k, v in outlier_tbl.iterrows()}

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, c in zip(axes, ["age", "balance", "duration", "campaign"]):
    sns.boxplot(y=df[c], color=BLUE, fliersize=2, ax=ax); ax.set_title(c); ax.set_ylabel("")
fig.suptitle("Outlier check: which variables have extreme values?", fontweight="bold")
fig.tight_layout(); save_fig(fig, "02_outlier_boxplots.png"); plt.show()

print(f"Negative balances: {(df['balance'] < 0).sum():,} ({(df['balance'] < 0).mean():.1%})   |   balance > €50,000: {(df['balance'] > 50000).sum()}")
print(f"Clients contacted 20+ times in this campaign: {(df['campaign'] >= 20).sum()}   |   calls longer than 1 hour: {(df['duration'] > 3600).sum()}")

**Decision:** extreme values (very high balances, many contacts, very long calls) are plausible real-world cases, not data-entry errors,
so they are **kept**. Negative balances are valid (overdrawn accounts). Because these variables are right-skewed, charts use clipped
ranges or log scales, and comparisons use **medians** as well as means.

## 5. Target variable analysis (Q1)

**Question:** what share of customers subscribed? The answer tells us the baseline every segment is compared against, and whether the classes are
imbalanced.

In [ ]:
counts = df["y"].value_counts()
overall = df["y_bin"].mean() * 100
print(f"Subscribed: {counts.get('yes', 0):,} of {len(df):,}  ->  {overall:.2f}%   |   not subscribed: {counts.get('no', 0):,} ({100 - overall:.2f}%)")
results.update(n_yes=int(counts.get("yes", 0)), overall_rate=float(overall))

fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.countplot(data=df, x="y", order=["no", "yes"], palette=[GREY, BLUE], hue="y", legend=False, ax=ax)
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{int(p.get_height()):,}\n({p.get_height() / len(df):.1%})", (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, counts.max() * 1.18); ax.set_xlabel("Subscribed to a term deposit?"); ax.set_ylabel("Customers")
ax.set_title("Only a minority of contacts subscribe")
save_fig(fig, "01_target_countplot.png"); plt.show()

**What it means:** the target is **imbalanced** - most contacts say no. A campaign that "predicts no for everyone" would already be
right most of the time, so overall accuracy would be a misleading metric and *lift* over the baseline rate is the useful measure.

## 6. Univariate analysis: distributions

How are the main numeric variables distributed? (Balance is clipped to the 1st-99th percentile for display; campaign uses a log scale.)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].hist(df["age"], bins=40, color=BLUE, edgecolor="white")
axes[0, 0].axvline(df["age"].median(), color=ORANGE, ls="--"); axes[0, 0].set_title("Age"); axes[0, 0].set_xlabel("Age (years)")
lo, hi = df["balance"].quantile([.01, .99])
b = df.loc[df["balance"].between(lo, hi), "balance"]
axes[0, 1].hist(b, bins=50, color=BLUE, edgecolor="white")
axes[0, 1].axvline(df["balance"].median(), color=ORANGE, ls="--"); axes[0, 1].set_title("Balance (1st-99th percentile)"); axes[0, 1].set_xlabel("Average yearly balance (€)")
d = df.loc[df["duration"] <= df["duration"].quantile(.99), "duration"]
axes[1, 0].hist(d, bins=50, color=BLUE, edgecolor="white")
axes[1, 0].axvline(df["duration"].median(), color=ORANGE, ls="--"); axes[1, 0].set_title("Call duration (up to 99th percentile)"); axes[1, 0].set_xlabel("Seconds")
cv = df["campaign"]
axes[1, 1].hist(cv, bins=np.logspace(0, np.log10(cv.max() + 1), 35), color=BLUE, edgecolor="white")
axes[1, 1].set_xscale("log"); axes[1, 1].axvline(cv.median(), color=ORANGE, ls="--"); axes[1, 1].set_title("Contacts in this campaign (log scale)"); axes[1, 1].set_xlabel("Number of contacts")
for ax in axes.flat: ax.set_ylabel("Customers")
fig.tight_layout(); save_fig(fig, "03_distributions.png"); plt.show()
display(df[["age", "balance", "duration", "campaign"]].describe(percentiles=[.5, .9, .99]).T)
results.update(median_age=float(df["age"].median()), median_balance=float(df["balance"].median()),
               median_duration=float(df["duration"].median()), median_campaign=float(df["campaign"].median()))

**What it means:** the dashed orange line marks the median. Age is fairly symmetric (most clients are in their 30s-50s), while balance,
call duration and number of contacts are **strongly right-skewed**: most clients have small values and a few have very large ones.
That is why medians and clipped/log views are used.

## 7. Who subscribes? Rates by customer characteristics (Q2-Q5)

For each customer group we plot the **subscription rate** (error bars = approximate 95% interval; labels show group size).
The dashed line is the overall rate. A bar clearly above the line is a group that responds better than average.

In [ ]:
overall = results["overall_rate"]

def rate_table(col, order=None):
    g = df.groupby(col, observed=True)["y_bin"].agg(["mean", "sum", "count"])
    g.columns = ["rate", "yes", "n"]; g["rate"] *= 100
    g = g.reindex(order) if order is not None else g.sort_values("rate", ascending=False)
    g["ci95"] = 1.96 * np.sqrt(g["rate"] / 100 * (1 - g["rate"] / 100) / g["n"]) * 100
    g["lift"] = g["rate"] / overall
    return g

def rate_bar(col, title, fname, order=None, horizontal=True, figsize=(9, 5.5), xlabel=None):
    g = rate_table(col, order)
    fig, ax = plt.subplots(figsize=figsize)
    labels = g.index.astype(str)
    if horizontal:
        gg, ll = g.iloc[::-1], labels[::-1]
        ax.barh(ll, gg["rate"], xerr=gg["ci95"], color=BLUE, height=0.68, error_kw=dict(ecolor="#555", lw=1, capsize=3))
        for i, (v, n) in enumerate(zip(gg["rate"], gg["n"])): ax.text(v + gg["ci95"].iloc[i] + 0.4, i, f"{v:.1f}%  (n={n:,})", va="center", fontsize=9, bbox=dict(facecolor="white", edgecolor="none", pad=1.5), zorder=5)
        ax.axvline(overall, color=ORANGE, ls="--", lw=1.6); ax.set_xlim(0, (gg["rate"] + gg["ci95"]).max() * 1.35)
        ax.set_xlabel("Subscription rate (%)"); ax.grid(axis="y", visible=False)
        ax.text(overall, len(g) - 0.4, f" overall {overall:.1f}%", color=ORANGE, fontsize=9, va="bottom")
    else:
        ax.bar(labels, g["rate"], yerr=g["ci95"], color=BLUE, width=0.65, error_kw=dict(ecolor="#555", lw=1, capsize=3))
        for i, (v, n) in enumerate(zip(g["rate"], g["n"])): ax.text(i, v + g["ci95"].iloc[i] + 0.5, f"{v:.1f}%\nn={n:,}", ha="center", fontsize=9)
        ax.axhline(overall, color=ORANGE, ls="--", lw=1.6); ax.set_ylim(0, (g["rate"] + g["ci95"]).max() * 1.3)
        ax.set_ylabel("Subscription rate (%)"); ax.set_xlabel(xlabel or ""); ax.grid(axis="x", visible=False)
        ax.text(len(g) - 0.5, overall, f"overall {overall:.1f}%", color=ORANGE, fontsize=9, va="bottom", ha="right")
    ax.set_title(title)
    save_fig(fig, fname); plt.show()
    return g

# Q2 age
age_t = rate_bar("age_group", "Subscription rate by age group", "04_rate_by_age.png",
                 order=["Under 25", "25–34", "35–44", "45–54", "55–64", "65+"], horizontal=False, figsize=(9, 5), xlabel="Age group")
display(age_t)
results["by_age"] = {str(k): [float(v["rate"]), int(v["n"])] for k, v in age_t.iterrows()}

In [ ]:
# Q3 job
job_t = rate_bar("job", "Subscription rate by job type", "05_rate_by_job.png", figsize=(9.5, 6.5))
display(job_t)
results["by_job"] = {str(k): [float(v["rate"]), int(v["n"])] for k, v in job_t.iterrows()}

# Q4 education
edu_t = rate_bar("education", "Subscription rate by education level", "06_rate_by_education.png", figsize=(9, 4.5))
display(edu_t)
results["by_education"] = {str(k): [float(v["rate"]), int(v["n"])] for k, v in edu_t.iterrows()}

# Q5 marital
mar_t = rate_bar("marital", "Subscription rate by marital status", "07_rate_by_marital.png", figsize=(9, 4))
display(mar_t)
results["by_marital"] = {str(k): [float(v["rate"]), int(v["n"])] for k, v in mar_t.iterrows()}

In [ ]:
# Cross-tabulations (row percentages): how each category splits into no / yes
for col in ["job", "education", "marital"]:
    print(f"\nSubscription outcome by {col} (row %):")
    display((pd.crosstab(df[col], df["y"], normalize="index") * 100).round(1).sort_values("yes", ascending=False))

**How to interpret:** compare each bar with the dashed overall line and look at the error bars and group size `n`. Small groups
(small `n`) have wide error bars, so their rates are less reliable. Job, age and education overlap (e.g. students are young, retired
clients are older), so a difference in one variable may partly reflect another - the segmentation section looks at combinations.

## 8. Account balance (Q6)

Do subscribers hold higher balances than non-subscribers? Balance is skewed, so we compare **medians** and use box plots without the extreme outliers.

In [ ]:
bal = df.groupby("y", observed=True)["balance"].agg(["count", "mean", "median", lambda s: s.quantile(.25), lambda s: s.quantile(.75)])
bal.columns = ["n", "mean", "median", "Q1", "Q3"]; display(bal)
results["balance_median_yes"] = float(bal.loc["yes", "median"]); results["balance_median_no"] = float(bal.loc["no", "median"])
results["balance_mean_yes"] = float(bal.loc["yes", "mean"]); results["balance_mean_no"] = float(bal.loc["no", "mean"])

band_t = rate_bar("balance_band", "Subscription rate by account balance band", "09_rate_by_balance_band.png",
                  order=["≤ €0", "€1–500", "€501–2,000", "€2,001–5,000", "> €5,000"], horizontal=False, figsize=(10, 5), xlabel="Average yearly balance")
results["by_balance_band"] = {str(k): [float(v["rate"]), int(v["n"])] for k, v in band_t.iterrows()}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.boxplot(data=df, x="y", y="balance", order=["no", "yes"], palette=[GREY, BLUE], hue="y", legend=False, showfliers=False, ax=axes[0])
axes[0].set_title("Balance by outcome (outliers hidden)"); axes[0].set_xlabel("Subscribed?"); axes[0].set_ylabel("Balance (€)")
sns.boxplot(data=df, x="y", y="age", order=["no", "yes"], palette=[GREY, BLUE], hue="y", legend=False, ax=axes[1])
axes[1].set_title("Age by outcome"); axes[1].set_xlabel("Subscribed?"); axes[1].set_ylabel("Age")
sns.boxplot(data=df[df["campaign"] <= 20], x="y", y="campaign", order=["no", "yes"], palette=[GREY, BLUE], hue="y", legend=False, showfliers=False, ax=axes[2])
axes[2].set_title("Contacts in this campaign by outcome"); axes[2].set_xlabel("Subscribed?"); axes[2].set_ylabel("Contacts")
fig.tight_layout(); save_fig(fig, "08_boxplots_by_outcome.png"); plt.show()

**What it means:** compare the medians (the line inside each box). A higher median balance for subscribers would suggest that customers
with more savings are better prospects - but note the boxes overlap heavily, so balance alone is a weak separator.

## 9. Campaign contacts and communication channel (Q7-Q8)

**Q7.** Do repeated contacts help or hurt? **Q8.** Which channel works best? (`unknown` contact type is a data gap, not a channel.)

In [ ]:
camp_t = rate_bar("campaign_group", "Subscription rate by number of contacts in this campaign", "10_rate_by_contacts.png",
                  order=["1", "2", "3", "4–5", "6–10", "11+"], horizontal=False, figsize=(9.5, 5), xlabel="Contacts made to the client in this campaign")
display(camp_t)
results["by_contacts"] = {str(k): [float(v["rate"]), int(v["n"])] for k, v in camp_t.iterrows()}

# Q8: contact channel - count plot (volume by outcome) + rate bars
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
order_c = df["contact"].value_counts().index
sns.countplot(data=df, x="contact", order=order_c, hue="y", hue_order=["no", "yes"], palette=[GREY, BLUE], ax=axes[0])
axes[0].set_title("Contacts by channel and outcome"); axes[0].set_xlabel("Contact channel"); axes[0].set_ylabel("Customers"); axes[0].legend(title="Subscribed")
ct = rate_table("contact")
axes[1].bar(ct.index.astype(str), ct["rate"], yerr=ct["ci95"], color=BLUE, width=0.6, error_kw=dict(ecolor="#555", capsize=3))
for i, (v, n) in enumerate(zip(ct["rate"], ct["n"])): axes[1].text(i, v + ct["ci95"].iloc[i] + 0.4, f"{v:.1f}%\nn={n:,}", ha="center", fontsize=9)
axes[1].axhline(overall, color=ORANGE, ls="--"); axes[1].set_ylim(0, (ct["rate"] + ct["ci95"]).max() * 1.3)
axes[1].set_title("Subscription rate by channel"); axes[1].set_xlabel("Contact channel"); axes[1].set_ylabel("Subscription rate (%)")
fig.tight_layout(); save_fig(fig, "11_contact_channel.png"); plt.show()
display(ct)
results["by_contact"] = {str(k): [float(v["rate"]), int(v["n"])] for k, v in ct.iterrows()}

**Reading the charts:** the left chart shows *volume* (how many calls used each channel and how they ended); the right chart shows *effectiveness*
(rate). A channel can be heavily used yet less effective. The `unknown` channel group should be interpreted with care because we do not know how those clients were reached.

## 10. Previous campaign history (Q9)

Does the result of an earlier campaign relate to today's outcome?

In [ ]:
pout_t = rate_table("poutcome")
display(pout_t)
results["by_poutcome"] = {str(k): [float(v["rate"]), int(v["n"])] for k, v in pout_t.iterrows()}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
order_p = ["success", "failure", "other", "unknown"]
order_p = [p for p in order_p if p in set(df["poutcome"])]
sns.countplot(data=df, x="poutcome", order=order_p, hue="y", hue_order=["no", "yes"], palette=[GREY, BLUE], ax=axes[0])
axes[0].set_yscale("log"); axes[0].set_title("Customers by previous outcome (log scale)"); axes[0].set_xlabel("Outcome of previous campaign"); axes[0].set_ylabel("Customers (log)")
axes[0].legend(title="Subscribed")
pt = pout_t.reindex(order_p)
axes[1].bar(pt.index.astype(str), pt["rate"], yerr=pt["ci95"], color=BLUE, width=0.6, error_kw=dict(ecolor="#555", capsize=3))
for i, (v, n) in enumerate(zip(pt["rate"], pt["n"])): axes[1].text(i, v + pt["ci95"].iloc[i] + 1, f"{v:.1f}%\nn={n:,}", ha="center", fontsize=9)
axes[1].axhline(overall, color=ORANGE, ls="--"); axes[1].set_ylim(0, (pt["rate"] + pt["ci95"]).max() * 1.25)
axes[1].set_title("Subscription rate by previous outcome"); axes[1].set_xlabel("Outcome of previous campaign"); axes[1].set_ylabel("Subscription rate (%)")
fig.tight_layout(); save_fig(fig, "12_previous_outcome.png"); plt.show()

pc = rate_table("previously_contacted"); display(pc)
results["by_previously_contacted"] = {str(k): [float(v["rate"]), int(v["n"])] for k, v in pc.iterrows()}

**What it means:** clients who said yes in a previous campaign are far more likely to say yes again, while `unknown` mostly means "never contacted
before". Previous history is therefore one of the strongest practical signals available *before* a call is made.

## 11. Time pattern: seasonality by month

The data only gives the month and day of the last contact (no year), so we can look at **seasonality across the calendar**, not at a long-term trend.
Months with few calls can show extreme rates, so group sizes are shown.

In [ ]:
mt = df.groupby("month", observed=True)["y_bin"].agg(["mean", "count"]); mt["rate"] = mt["mean"] * 100
mt = mt.reindex([m for m in MONTHS if m in mt.index])
display(mt[["count", "rate"]])
results["by_month"] = {k: [float(v["rate"]), int(v["count"])] for k, v in mt.iterrows()}

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]})
axes[0].bar(mt.index.astype(str), mt["count"], color=GREY, width=0.65); axes[0].set_ylabel("Contacts"); axes[0].set_title("When does the bank call, and when do customers say yes?")
axes[1].plot(mt.index.astype(str), mt["rate"], color=BLUE, lw=2.4, marker="o")
axes[1].axhline(overall, color=ORANGE, ls="--"); axes[1].text(0, overall, f" overall {overall:.1f}%", color=ORANGE, va="bottom", fontsize=9)
best_m = mt[mt["count"] >= 200]["rate"].idxmax()
axes[1].annotate(f"best (n≥200): {best_m} {mt.loc[best_m, 'rate']:.0f}%", (list(mt.index).index(best_m), mt.loc[best_m, "rate"]), textcoords="offset points", xytext=(8, 6), fontsize=9)
axes[1].set_ylabel("Subscription rate (%)"); axes[1].set_xlabel("Month of last contact"); axes[1].set_ylim(bottom=0)
fig.tight_layout(); save_fig(fig, "13_monthly_pattern.png"); plt.show()
results["best_month"] = best_m; results["busiest_month"] = mt["count"].idxmax()

**What it means:** call volume is very uneven (a few months account for most calls) and the months with the **most** calls are not the months
with the **best** conversion. That mismatch is a planning opportunity, with the caveat that quiet-month customers may be a different (more selected) group.

## 12. Bivariate and multivariate analysis

### Scatter plots (relationships between two numeric variables, coloured by outcome)

In [ ]:
samp = df.sample(min(8000, len(df)), random_state=42)
lo, hi = df["balance"].quantile([.01, .99])
s2 = samp[samp["balance"].between(lo, hi)]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for lab, col, a in [("no", GREY, 0.35), ("yes", ORANGE, 0.65)]:
    d = s2[s2["y"] == lab]; axes[0].scatter(d["age"], d["balance"], s=10, alpha=a, color=col, label=lab, linewidths=0)
axes[0].set_title("Age vs balance (random sample)"); axes[0].set_xlabel("Age"); axes[0].set_ylabel("Balance € (1st-99th percentile)"); axes[0].legend(title="Subscribed", markerscale=2)
s3 = samp[(samp["duration"] <= samp["duration"].quantile(.99)) & (samp["campaign"] <= 20)]
for lab, col, a in [("no", GREY, 0.35), ("yes", ORANGE, 0.65)]:
    d = s3[s3["y"] == lab]; axes[1].scatter(d["duration"], d["campaign"] + np.random.default_rng(1).uniform(-.25, .25, len(d)), s=10, alpha=a, color=col, label=lab, linewidths=0)
axes[1].set_title("Call duration vs number of contacts"); axes[1].set_xlabel("Call duration (seconds)"); axes[1].set_ylabel("Contacts in this campaign"); axes[1].legend(title="Subscribed", markerscale=2)
fig.tight_layout(); save_fig(fig, "14_scatter_plots.png"); plt.show()

dur = df.groupby("y", observed=True)["duration"].median()
results.update(duration_median_yes=float(dur.get("yes", np.nan)), duration_median_no=float(dur.get("no", np.nan)))
print(f"Median call duration: subscribers {dur.get('yes'):.0f}s vs non-subscribers {dur.get('no'):.0f}s")

**Important caveat about `duration`:** subscribers tend to have longer calls, but call duration is **only known after the call ends**, and a client who is
interested naturally talks longer. It cannot be used to decide whom to call, so we use it to understand the conversation, not for targeting.

In [ ]:
# Correlation heatmap (Pearson; with the 0/1 target this is a point-biserial correlation)
num = df[["age", "balance", "day", "duration", "campaign", "pdays_clean", "previous", "y_bin"]].rename(columns={"y_bin": "subscribed"})
corr = num.corr()
fig, ax = plt.subplots(figsize=(8.5, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0, linewidths=.5, square=True, ax=ax)
ax.set_title("Correlation heatmap of numeric variables and the target")
fig.tight_layout(); save_fig(fig, "15_correlation_heatmap.png"); plt.show()

target_corr = corr["subscribed"].drop("subscribed").sort_values(key=abs, ascending=False)
display(target_corr.to_frame("correlation with subscription"))
results["target_corr"] = {k: float(v) for k, v in target_corr.items()}

**What it means:** no single numeric variable is strongly correlated with subscription (all coefficients are small to moderate), which tells us the outcome depends
on **combinations** of factors and on categorical drivers such as job and previous outcome. `pdays_clean` is NaN for clients never contacted before, so its
correlation uses only previously-contacted clients.

## 13. Customer segmentation (Q10)

We build simple, explainable **rule-based segments** from age group, balance band and housing-loan status, then look for segments that
respond much better or worse than average. Only segments with **at least 300 customers** are ranked, so results are not driven by tiny groups.

In [ ]:
# Heatmap: age group x balance band
pivot_rate = df.pivot_table(index="age_group", columns="balance_band", values="y_bin", aggfunc="mean", observed=True) * 100
pivot_n = df.pivot_table(index="age_group", columns="balance_band", values="y_bin", aggfunc="count", observed=True)
masked = pivot_rate.where(pivot_n >= 100)
annot = masked.round(1).astype(object).where(masked.notna(), "")
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(masked, annot=annot, fmt="", cmap="Blues", linewidths=.5, linecolor="white", cbar_kws={"label": "Subscription rate (%)"}, ax=ax)
ax.set_title("Subscription rate by age group and balance band (cells with n < 100 hidden)"); ax.set_xlabel("Balance band"); ax.set_ylabel("Age group")
save_fig(fig, "16_segment_heatmap.png"); plt.show()

# Rule-based segments
df["segment"] = (df["age_group"].astype(str) + " | " + df["balance_band"].astype(str) + " | " +
                 np.where(df["housing"] == "yes", "housing loan", "no housing loan"))
seg = df.groupby("segment", observed=True)["y_bin"].agg(["mean", "sum", "count"]); seg.columns = ["rate", "yes", "n"]; seg["rate"] *= 100
seg["lift"] = seg["rate"] / overall
seg = seg[seg["n"] >= 300].sort_values("rate", ascending=False)
print(f"{len(seg)} segments with n >= 300 (overall rate {overall:.1f}%)")
print("Top 10 segments:"); display(seg.head(10))
print("Bottom 5 segments:"); display(seg.tail(5))
results["top_segments"] = [(k, float(v["rate"]), int(v["n"]), float(v["lift"])) for k, v in seg.head(5).iterrows()]
results["bottom_segments"] = [(k, float(v["rate"]), int(v["n"]), float(v["lift"])) for k, v in seg.tail(3).iterrows()]

sel = pd.concat([seg.head(8), seg.tail(4)])
fig, ax = plt.subplots(figsize=(11, 6))
d = sel.iloc[::-1]
ax.barh(d.index, d["rate"], color=[BLUE if r >= overall else ORANGE for r in d["rate"]], height=0.7)
for i, (r_, n_) in enumerate(zip(d["rate"], d["n"])): ax.text(r_ + 0.3, i, f"{r_:.1f}%  (n={n_:,})", va="center", fontsize=9)
ax.axvline(overall, color="#333", ls="--", lw=1.4); ax.set_xlim(0, d["rate"].max() * 1.25); ax.grid(axis="y", visible=False)
ax.set_title("Best and worst customer segments (age | balance | housing loan)"); ax.set_xlabel("Subscription rate (%)")
save_fig(fig, "17_top_segments.png"); plt.show()

**What it means:** segments at the top are the most promising audiences for a focused campaign; segments at the bottom are costly to call for little return.
Segments are *descriptive* groupings of this data, not proven causes - they should be tested in a pilot before being used to target at scale.

## 14. Insights, recommendations and limitations

### Cross-check against the standard published dataset
These are sanity checks that you have loaded the standard `bank-full.csv` file (not targets):

In [ ]:
def chk(label, expected, actual, tol=0):
    return {"check": label, "expected": expected, "calculated": actual, "result": "match" if abs(expected - actual) <= tol else "DIFFERENT"}
r = results
check_df = pd.DataFrame([
    chk("Rows", 45211, r["raw_rows"]), chk("Columns", 17, r["raw_cols"]), chk("Duplicate rows", 0, r["duplicate_rows"]),
    chk("Missing (NaN) cells", 0, r["missing_cells"]), chk("Subscribed customers", 5289, r["n_yes"]),
    chk("Subscription rate (%)", 11.70, r["overall_rate"], 0.01)])
display(check_df)
results["n_checks_match"] = int((check_df["result"] == "match").sum()); results["n_checks"] = len(check_df)
if (check_df["result"] != "match").any():
    print("Some values differ: you may have loaded a different file (e.g. the 4,521-row sample bank.csv, which is a 10% subset of bank-full.csv).")

In [ ]:
r = results
def best(d, min_n=300):      # best category among reasonably large groups
    items = [(k, v[0], v[1]) for k, v in d.items() if v[1] >= min_n]; return max(items, key=lambda t: t[1])
def worst(d, min_n=300):
    items = [(k, v[0], v[1]) for k, v in d.items() if v[1] >= min_n]; return min(items, key=lambda t: t[1])

age_b, age_w = best(r["by_age"]), worst(r["by_age"])
job_b, job_w = best(r["by_job"]), worst(r["by_job"])
edu_b, edu_w = best(r["by_education"]), worst(r["by_education"])
mar_b, mar_w = best(r["by_marital"]), worst(r["by_marital"])
con_b = best({k: v for k, v in r["by_contact"].items() if k != "unknown"}, 100)
camp = r["by_contacts"]; camp_keys = list(camp.keys())
pout = r["by_poutcome"]
m = r["by_month"]
top_seg = r["top_segments"][0]

recs = []
recs.append(f"Prioritise high-propensity audiences: the best segment ({top_seg[0]}) subscribes at {top_seg[1]:.1f}% versus {r['overall_rate']:.1f}% overall (lift {top_seg[3]:.1f}x, n={top_seg[2]:,}). Pilot campaigns on segments like this first.")
recs.append(f"Target by age and occupation: {age_b[0]} is the strongest age group ({age_b[1]:.1f}%) and '{job_b[0]}' the strongest job group ({job_b[1]:.1f}%), while '{job_w[0]}' is the weakest ({job_w[1]:.1f}%). Shift call effort towards the former and use cheaper channels for the latter.")
if "success" in pout:
    recs.append(f"Re-contact previous successes first: clients with a successful earlier campaign subscribe at {pout['success'][0]:.1f}% versus {pout.get('unknown', [overall])[0]:.1f}% for clients with no recorded history (n={pout['success'][1]:,}).")
recs.append(f"Cap repeated contacts: the subscription rate is {camp[camp_keys[0]][0]:.1f}% at one contact and {camp[camp_keys[-1]][0]:.1f}% at {camp_keys[-1]} contacts. Set a contact limit and stop calling unresponsive clients to save cost.")
recs.append(f"Prefer the more effective channel: {con_b[0]} contacts convert at {con_b[1]:.1f}%; improve recording of the contact channel because 'unknown' ({r['by_contact'].get('unknown', [np.nan])[0]:.1f}%) hides what was used.")
recs.append(f"Schedule by season: the best month for conversion (n≥200) is {r['best_month']} while the busiest month for calls is {r['busiest_month']}. Test moving some call capacity towards high-conversion months.")
if r["balance_median_yes"] > 1.15 * r["balance_median_no"]:
    recs.append(f"Use balance as a screening aid: median balance is €{r['balance_median_yes']:,.0f} for subscribers vs €{r['balance_median_no']:,.0f} for non-subscribers, but the overlap is large so combine it with other signals.")
else:
    recs.append(f"Do not rely on balance alone: median balance is similar for subscribers (€{r['balance_median_yes']:,.0f}) and non-subscribers (€{r['balance_median_no']:,.0f}), so use job, previous outcome and contact history instead.")
recs.append(f"Do not use call duration for targeting (median {r['duration_median_yes']:.0f}s for subscribers vs {r['duration_median_no']:.0f}s): it is only known after the call. Use it to evaluate call scripts and agent training instead.")
results["recommendations"] = recs
print("KEY INSIGHTS")
print(f"- {r['overall_rate']:.1f}% of {r['clean_rows']:,} contacts subscribed (imbalanced target).")
print(f"- Age: best {age_b[0]} ({age_b[1]:.1f}%), lowest {age_w[0]} ({age_w[1]:.1f}%).")
print(f"- Job: best '{job_b[0]}' ({job_b[1]:.1f}%), lowest '{job_w[0]}' ({job_w[1]:.1f}%).")
print(f"- Education: best {edu_b[0]} ({edu_b[1]:.1f}%), lowest {edu_w[0]} ({edu_w[1]:.1f}%).")
print(f"- Marital status: best {mar_b[0]} ({mar_b[1]:.1f}%), lowest {mar_w[0]} ({mar_w[1]:.1f}%).")
print()
print("RECOMMENDATIONS")
for i, t in enumerate(recs, 1): print(f"{i}. {t}\n")
results.update(age_best=age_b, age_worst=age_w, job_best=job_b, job_worst=job_w, edu_best=edu_b, edu_worst=edu_w, mar_best=mar_b, mar_worst=mar_w, contact_best=con_b)

### Limitations

* **Association, not causation.** Group differences describe this campaign data; they do not prove that a characteristic *causes* subscription.
* **Imbalanced target** (most contacts say no), so overall accuracy-style thinking is misleading; we use rates and lift.
* **`unknown` labels** in `poutcome`, `contact`, `education` and `job` hide missing information and may cover different kinds of customers.
* **`duration` is known only after a call**, so it cannot be used for targeting (leakage risk if used in a predictive model).
* **No year in the data**, so seasonality cannot be separated from long-term trends; months with few calls give less reliable rates.
* **One Portuguese bank, 2008-2010 campaigns**; the findings may not apply to other banks, countries or periods.
* Segments overlap and are rule-based; they should be validated with a pilot campaign.

## 15. Save results, EDA report (PDF) and presentation (PowerPoint)

The cells below write `reports/results.json`, `reports/EDA_report.pdf`, `reports/final_report.md` and `reports/presentation.pptx`
(8 slides). They need `reportlab` and `python-pptx` (both are in `requirements.txt`). All numbers come from the calculations above.

In [ ]:
import html
from PIL import Image as PILImage

def build_pdf(path, title, subtitle, sections):
    """sections = [{"heading": str, "paras": [str], "images": [filename in reports/]}]"""
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak, KeepTogether
    ss = getSampleStyleSheet()
    H1 = ParagraphStyle("H1", parent=ss["Title"], fontSize=22, textColor=colors.HexColor("#1b2a41"), spaceAfter=6)
    H2 = ParagraphStyle("H2", parent=ss["Heading2"], textColor=colors.HexColor("#1b2a41"), spaceBefore=10, spaceAfter=4)
    B = ParagraphStyle("B", parent=ss["BodyText"], fontSize=10.5, leading=15, spaceAfter=5)
    SUB = ParagraphStyle("SUB", parent=B, textColor=colors.HexColor("#555555"), alignment=1)
    story = [Spacer(1, 3 * cm), Paragraph(html.escape(title), H1), Paragraph(html.escape(subtitle), SUB), Spacer(1, 1 * cm)]
    for sec in sections:
        block = [Paragraph(html.escape(sec["heading"]), H2)]
        for p in sec.get("paras", []):
            block.append(Paragraph(html.escape(p), B))
        story.append(KeepTogether(block[:2]) if len(block) > 1 else block[0])
        story.extend(block[2:])
        for img in sec.get("images", []):
            fp = REPORTS_DIR / img
            if fp.exists():
                w, h = PILImage.open(fp).size
                width = 16.5 * cm; height = width * h / w
                if height > 11 * cm:
                    height = 11 * cm; width = height * w / h
                story.extend([Image(str(fp), width=width, height=height), Spacer(1, 6)])
    doc = SimpleDocTemplate(str(path), pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm, topMargin=1.8 * cm, bottomMargin=1.8 * cm,
                            title=title)
    doc.build(story)

def build_pptx(path, title, subtitle, slides):
    """slides = [{"title": str, "bullets": [str], "images": [filename]}]  (title slide is added first)"""
    from pptx import Presentation
    from pptx.util import Inches, Pt
    from pptx.dml.color import RGBColor
    NAVY, TEXT = RGBColor(0x1b, 0x2a, 0x41), RGBColor(0x33, 0x33, 0x33)
    prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)

    def tb(s, x, y, w, h, lines, size, bold=False, color=TEXT):
        box = s.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h)); tf = box.text_frame; tf.word_wrap = True
        for i, line in enumerate(lines if isinstance(lines, list) else [lines]):
            p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
            p.text = line; p.font.size = Pt(size); p.font.bold = bold; p.font.color.rgb = color; p.space_after = Pt(8)

    def place(s, name, x, y, w, h):
        fp = REPORTS_DIR / name
        if not fp.exists(): return
        iw, ih = PILImage.open(fp).size
        scale = min(w / iw, h / ih); pw, ph = iw * scale, ih * scale
        s.shapes.add_picture(str(fp), Inches(x + (w - pw) / 2), Inches(y), Inches(pw), Inches(ph))

    s = prs.slides.add_slide(prs.slide_layouts[6])
    tb(s, 0.8, 2.0, 11.7, 1.6, title, 38, True, NAVY); tb(s, 0.8, 3.8, 11.7, 1.5, subtitle, 18)
    for sl in slides:
        s = prs.slides.add_slide(prs.slide_layouts[6])
        tb(s, 0.6, 0.3, 12.1, 1.0, sl["title"], 26, True, NAVY)
        imgs, bullets = sl.get("images", []), sl.get("bullets", [])
        if len(imgs) == 1:
            place(s, imgs[0], 0.4, 1.4, 8.2, 5.6); tb(s, 8.8, 1.6, 4.2, 5.4, bullets, 15)
        elif len(imgs) >= 2:
            place(s, imgs[0], 0.3, 1.4, 6.4, 3.9); place(s, imgs[1], 6.7, 1.4, 6.4, 3.9); tb(s, 0.6, 5.5, 12.1, 1.8, bullets, 15)
        else:
            tb(s, 0.8, 1.5, 11.7, 5.5, bullets, 20)
    prs.save(path)

In [ ]:
with open(REPORTS_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, default=str)

r = results
TITLE = "Bank Marketing Campaign & Customer Segmentation Analysis"
SUB = f"UCI Bank Marketing dataset | {r['raw_rows']:,} customer contacts | target: term-deposit subscription"
f1 = lambda t: f"{t[0]} ({t[1]:.1f}%)"

sections = [
 {"heading": "1. Objective and data", "paras": [
   f"A bank wants to make its telephone marketing for term deposits more effective. The dataset has {r['raw_rows']:,} contacts and {r['raw_cols']} columns. The target is whether the client subscribed.",
   f"Cleaning: {r['duplicate_rows']:,} duplicate rows and {r['missing_cells']:,} missing cells were found. The label 'unknown' (e.g. in poutcome and contact) is a hidden missing value and was kept as its own category. pdays = -1 (never contacted) was converted to a missing value for statistics. Outliers in balance, duration and contacts are plausible real cases and were kept; medians and log/clipped views are used."]},
 {"heading": "2. Target and distributions", "paras": [
   f"{r['overall_rate']:.2f}% of contacts subscribed ({r['n_yes']:,} of {r['clean_rows']:,}), so the target is imbalanced. Median age is {r['median_age']:.0f}, median balance €{r['median_balance']:,.0f}, median call {r['median_duration']:.0f}s and median contacts {r['median_campaign']:.0f}; balance, duration and contacts are strongly right-skewed."],
  "images": ["01_target_countplot.png", "03_distributions.png", "02_outlier_boxplots.png"]},
 {"heading": "3. Age, job, education and marital status", "paras": [
   f"Age: highest rate {f1(r['age_best'])}, lowest {f1(r['age_worst'])}. Job: highest '{r['job_best'][0]}' ({r['job_best'][1]:.1f}%), lowest '{r['job_worst'][0]}' ({r['job_worst'][1]:.1f}%). Education: highest {f1(r['edu_best'])}, lowest {f1(r['edu_worst'])}. Marital status: highest {f1(r['mar_best'])}, lowest {f1(r['mar_worst'])}. Group sizes differ, and these variables overlap, so differences are associations only."],
  "images": ["04_rate_by_age.png", "05_rate_by_job.png", "06_rate_by_education.png", "07_rate_by_marital.png"]},
 {"heading": "4. Balance", "paras": [
   f"Median balance is €{r['balance_median_yes']:,.0f} for subscribers and €{r['balance_median_no']:,.0f} for non-subscribers (means €{r['balance_mean_yes']:,.0f} vs €{r['balance_mean_no']:,.0f}). The distributions overlap heavily."],
  "images": ["09_rate_by_balance_band.png", "08_boxplots_by_outcome.png"]},
 {"heading": "5. Campaign contacts and channel", "paras": [
   f"The subscription rate by number of contacts falls from {r['by_contacts']['1'][0]:.1f}% at one contact to {list(r['by_contacts'].values())[-1][0]:.1f}% for the highest group. The most effective known channel is {r['contact_best'][0]} ({r['contact_best'][1]:.1f}%)."],
  "images": ["10_rate_by_contacts.png", "11_contact_channel.png"]},
 {"heading": "6. Previous campaign and seasonality", "paras": [
   "Clients with a successful previous campaign respond far more often than average." if "success" in r["by_poutcome"] else "See the previous-outcome chart.",
   f"Best month by conversion (n≥200): {r['best_month']}; busiest month by call volume: {r['busiest_month']}. The data has no year, so only calendar seasonality can be assessed."],
  "images": ["12_previous_outcome.png", "13_monthly_pattern.png"]},
 {"heading": "7. Correlations and relationships", "paras": [
   "No single numeric variable is strongly correlated with subscription, so outcomes depend on combinations of factors. Call duration is longer for subscribers but is only known after the call, so it cannot be used for targeting."],
  "images": ["15_correlation_heatmap.png", "14_scatter_plots.png"]},
 {"heading": "8. Customer segmentation", "paras": [
   f"Best segment: {r['top_segments'][0][0]} at {r['top_segments'][0][1]:.1f}% (n={r['top_segments'][0][2]:,}, lift {r['top_segments'][0][3]:.1f}x). Lowest: {r['bottom_segments'][-1][0]} at {r['bottom_segments'][-1][1]:.1f}% (n={r['bottom_segments'][-1][2]:,}). Only segments with at least 300 customers are ranked."],
  "images": ["16_segment_heatmap.png", "17_top_segments.png"]},
 {"heading": "9. Recommendations", "paras": r["recommendations"]},
 {"heading": "10. Limitations", "paras": [
   "Findings are associations, not causes. The target is imbalanced. 'unknown' labels hide missing information. Duration is known only after the call. The data has no year. One Portuguese bank, 2008-2010; results may not transfer. Segments are rule-based and should be validated with a pilot campaign."]},
 {"heading": "11. Dataset check", "paras": [f"{r['n_checks_match']} of {r['n_checks']} sanity checks against the standard bank-full.csv matched."]},
]
build_pdf(REPORTS_DIR / "EDA_report.pdf", TITLE, SUB, sections)
print("Saved ->", (REPORTS_DIR / "EDA_report.pdf").relative_to(PROJECT_DIR))
md_txt = f"# {TITLE}\n\n*{SUB}*\n\n" + "\n\n".join(f"## {s['heading']}\n\n" + "\n\n".join(s.get("paras", [])) for s in sections)
(REPORTS_DIR / "final_report.md").write_text(md_txt, encoding="utf-8")

slides = [
 {"title": "Who subscribes? Age and job matter", "images": ["04_rate_by_age.png", "05_rate_by_job.png"],
  "bullets": [f"Best age group: {f1(r['age_best'])}; lowest: {f1(r['age_worst'])}", f"Best job: {r['job_best'][0]} ({r['job_best'][1]:.1f}%); lowest: {r['job_worst'][0]} ({r['job_worst'][1]:.1f}%)", f"Education: best {f1(r['edu_best'])}; marital: best {f1(r['mar_best'])}"]},
 {"title": "Balance and number of contacts", "images": ["09_rate_by_balance_band.png", "10_rate_by_contacts.png"],
  "bullets": [f"Median balance €{r['balance_median_yes']:,.0f} (subscribers) vs €{r['balance_median_no']:,.0f} (others): large overlap", f"Success rate falls from {r['by_contacts']['1'][0]:.1f}% at 1 contact to {list(r['by_contacts'].values())[-1][0]:.1f}% at the highest group"]},
 {"title": "Which channel works best?", "images": ["11_contact_channel.png"],
  "bullets": [f"Best known channel: {r['contact_best'][0]} ({r['contact_best'][1]:.1f}%)", "The 'unknown' channel is a data gap, not a channel", "Left: volume by outcome; right: subscription rate"]},
 {"title": "Previous campaign history is a strong signal", "images": ["12_previous_outcome.png"],
  "bullets": [(f"Previous success converts at {r['by_poutcome']['success'][0]:.1f}% vs {r['overall_rate']:.1f}% overall" if "success" in r["by_poutcome"] else "Previous history relates to the current outcome"), "'unknown' mostly means never contacted before"]},
 {"title": "Seasonality: busiest month is not the best month", "images": ["13_monthly_pattern.png"],
  "bullets": [f"Busiest month: {r['busiest_month']}", f"Best conversion month (n≥200): {r['best_month']}", "No year in data: seasonality only"]},
 {"title": "Customer segments", "images": ["17_top_segments.png"],
  "bullets": [f"Best segment: {r['top_segments'][0][0]} - {r['top_segments'][0][1]:.1f}% (lift {r['top_segments'][0][3]:.1f}x)", "No single numeric variable strongly predicts subscription: combinations matter", "Segments with n < 300 are ignored"]},
 {"title": "Recommendations and limitations", "bullets": [t.split(":")[0] for t in r["recommendations"][:6]] + ["Limitations: associations not causes; 'unknown' values; duration known after call; one bank, 2008-2010"]},
]
SUB_PPT = SUB + f" | only {r['overall_rate']:.1f}% subscribed (imbalanced target) | Python, pandas, NumPy, Matplotlib, Seaborn"
build_pptx(REPORTS_DIR / "presentation.pptx", TITLE, SUB_PPT, slides)
print("Saved ->", (REPORTS_DIR / "presentation.pptx").relative_to(PROJECT_DIR))